In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

In [17]:
data = pd.read_csv("../../Data/processed_forecast_data.csv")


data.head()

,Date,Song,Artist,Rank,Last Week,Weeks in Charts,Normalized Title,Danceability,Energy,Key,...,Duration_ms,Time_signature,Chorus_hit,Sections,rank_change,target_next_rank,rolling_avg_4_weeks,rolling_avg_all_time,rolling_std_4_weeks,rolling_std_all_time
0,1958-12-17,Jingle Bell Rock,Bobby Helms,57,NaN,NaN,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,NaN,35.0,57.000000,57.000000,NaN,NaN
1,1958-12-24,Jingle Bell Rock,Bobby Helms,35,57.0,2.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,22.0,45.0,46.000000,46.000000,15.556349,15.556349
2,1958-12-31,Jingle Bell Rock,Bobby Helms,45,35.0,3.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,-10.0,70.0,45.666667,45.666667,11.015141,11.015141
3,1959-01-07,Jingle Bell Rock,Bobby Helms,70,45.0,4.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,-25.0,69.0,51.750000,51.750000,15.129992,15.129992
4,1960-12-07,Rockin' Around The Christmas Tree,Brenda Lee,64,NaN,NaN,rockin around the christmas tree,0.589,0.472,8,...,126267,4,33.91817,6,NaN,26.0,64.000000,64.000000,NaN,NaN


In [18]:
# Currently reduce audio features to the most important ones based on feature importance from previous classification model, to see if it improves forecasting performance. Will experiment with adding more features later.
audio_features = ['Instrumentalness', 'Danceability', 'Acousticness', 'Duration_ms', 'Energy', 'Valence']
all_features = ['Date', 'Song', 'Artist', 'Rank', 'Last Week', 'Weeks in Charts', 'Normalized Title', 'rank_change', 'rolling_avg_4_weeks', 'rolling_avg_all_time', 'rolling_std_4_weeks', 'rolling_std_all_time', 'target_next_rank'] + audio_features

In [19]:
data = data[all_features]

In [20]:
data.head()

,Date,Song,Artist,Rank,Last Week,Weeks in Charts,Normalized Title,rank_change,rolling_avg_4_weeks,rolling_avg_all_time,rolling_std_4_weeks,rolling_std_all_time,target_next_rank,Instrumentalness,Danceability,Acousticness,Duration_ms,Energy,Valence
0,1958-12-17,Jingle Bell Rock,Bobby Helms,57,NaN,NaN,jingle bell rock,NaN,57.000000,57.000000,NaN,NaN,35.0,0.0,0.754,0.643,130973,0.424,0.806
1,1958-12-24,Jingle Bell Rock,Bobby Helms,35,57.0,2.0,jingle bell rock,22.0,46.000000,46.000000,15.556349,15.556349,45.0,0.0,0.754,0.643,130973,0.424,0.806
2,1958-12-31,Jingle Bell Rock,Bobby Helms,45,35.0,3.0,jingle bell rock,-10.0,45.666667,45.666667,11.015141,11.015141,70.0,0.0,0.754,0.643,130973,0.424,0.806
3,1959-01-07,Jingle Bell Rock,Bobby Helms,70,45.0,4.0,jingle bell rock,-25.0,51.750000,51.750000,15.129992,15.129992,69.0,0.0,0.754,0.643,130973,0.424,0.806
4,1960-12-07,Rockin' Around The Christmas Tree,Brenda Lee,64,NaN,NaN,rockin around the christmas tree,NaN,64.000000,64.000000,NaN,NaN,26.0,0.0,0.589,0.614,126267,0.472,0.898


In [ ]:
#Check final date in the dataset
data['Date'] = pd.to_datetime(data['Date'])
print(data['Date'].max())
#Split data into train and test based on date, using 2014 as the cutoff date, as we want to forecast 2015 to 2025
train_data = data[data['Date'] < '2014-01-01']
test_data = data[data['Date'] >= '2014-01-01']
print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"Train data is {round(len(train_data)/len(data)*100, 2)}% of the total data")


2025-07-30 00:00:00
Train data shape: (105382, 19)
Test data shape: (27083, 19)
Train data is 79.55% of the total data


FIRST XGBOOST

In [42]:
features = [
    'Rank',
    'Last Week',
    'Weeks in Charts',
] + audio_features

In [43]:
y = data['target_next_rank']
X = data[features]

In [46]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)